In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import pycountry
from scipy import stats
from scipy.signal import savgol_filter

# Cargar datos de proyeccion PIB
Proyeccion_PIB = '../../data/fuentes/economicos/Proyeccion_PIB_indicepais.xlsx'
df_proyPib = pd.read_excel(Proyeccion_PIB, sheet_name='Impacto en PIB')


#Trasponer columnas de escenarios

df_proyPibtras=df_proyPib.melt(
    id_vars = ["Pais/ Region", "Escenario incremento T°"],
    value_vars= [
        "Sin incremento en desastres",
        "Mediano incremento en desastres (x5)",
        "Extremo incremento en desastres (x10)"
    ],

    var_name= "Escenario desastres",
    value_name= "Impacto en PIB"
)

# Transformar en formato número

df_proyPibtras["Impacto en PIB"] = (
        df_proyPibtras["Impacto en PIB"].astype(str)
        .str.replace("–", "-", regex=False)           
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
)

df_proyPibtras["Impacto en PIB"]=pd.to_numeric(df_proyPibtras["Impacto en PIB"], errors="coerce")

print(df_proyPibtras.head(10))

           Pais/ Region              Escenario incremento T°  \
0                 World  Incremento bajo 2°C (Acuerdo París)   
1                  OECD  Incremento bajo 2°C (Acuerdo París)   
2         North America  Incremento bajo 2°C (Acuerdo París)   
3         South America  Incremento bajo 2°C (Acuerdo París)   
4                Europe  Incremento bajo 2°C (Acuerdo París)   
5  Middle East & Africa  Incremento bajo 2°C (Acuerdo París)   
6                  Asia  Incremento bajo 2°C (Acuerdo París)   
7         Advanced Asia  Incremento bajo 2°C (Acuerdo París)   
8                 ASEAN  Incremento bajo 2°C (Acuerdo París)   
9               Oceania  Incremento bajo 2°C (Acuerdo París)   

           Escenario desastres  Impacto en PIB  
0  Sin incremento en desastres            -0.5  
1  Sin incremento en desastres            -0.4  
2  Sin incremento en desastres            -0.5  
3  Sin incremento en desastres            -0.4  
4  Sin incremento en desastres            -0.2  
5

In [2]:
import pandas as pd
import numpy as np
import pymysql
from pymysql.constants import CLIENT
from dotenv import load_dotenv
import os

load_dotenv()

# Obtener los parámetros de conexión
DB_HOST = os.getenv('DB_HOST')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_NAME = os.getenv('DB_NAME')

conexion = pymysql.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
    client_flag=CLIENT.MULTI_STATEMENTS
)
cursor = conexion.cursor()

# 2) Cargar dimensión Paises: (codigo, nombre_en) → dict nombre_en_normalizado → codigo
cursor.execute("SELECT codigo, nombre_en FROM Paises;")
dim_paises = {
    nombre_en.strip().lower(): codigo
    for codigo, nombre_en in cursor.fetchall()
}



# 1) Normalizar columna "Pais/ Region" → minúsculas y sin espacios
df_proyPibtras["pais_norm"] = (
    df_proyPibtras["Pais/ Region"]
      .astype(str)
      .str.strip()
      .str.lower()
)

# 2) Diccionario de excepciones (adaptado a tus datos)
exceptions = {
    'netherlands':'netherlands (kingdom of the)',
    'turkiye':'türkiye',
    'united kingdom':'united kingdom of great britain and northern ireland',
    'bahamas, the':'bahamas',
    'bolivia':'bolivia (plurinational state of)',
    'congo, dem. rep.':'congo (the democratic republic of the)',
    'congo, rep.':'congo',
    "cote d'ivoire":"côte d'ivoire",
    'egypt, arab rep.':'egypt',
    'gambia, the':'gambia',
    'hong kong sar, china':'hong kong',
    'iran, islamic rep.':'iran (islamic republic of)',
    'korea, rep.':'korea (the republic of)',
    'micronesia, fed. states of':'micronesia (federated states of)',
    'st. vincent and the grenadines':'saint vincent and the grenadines',
    'tanzania':'tanzania, the united republic of',
    'curacao':'curaçao',
    "korea, dem. people's rep.":"korea (the democratic people's republic of)",
    'slovak republic':'slovakia',
    'venezuela, rb':'venezuela (bolivarian republic of)',
    'yemen, rep.':'yemen',
    'st. kitts and nevis':'saint kitts and nevis',
    'st. lucia':'saint lucia',
    'macao sar, china':'macao',
    'lao pdr':"lao people's democratic republic",
    'kyrgyz republic':'kyrgyzstan',
    'russian federation':'russian federation',
    'moldova':'moldova (the republic of)',
    'united states':'united states of america',
    'us':'united states of america',
    'st. martin (french part)':'saint martin (french part)',
    'british virgin islands':'virgin islands (british)',
    'venezuela':'Venezuela (Bolivarian Republic of)',
    'south korea':'Korea (the Republic of)',
    'czech republic': 'Czechia',
    'russia': 'Russian Federation',
    'taiwan': 'Taiwan (Province of China)',
    'turkey': 'Türkiye',
    'uae': 'United Arab Emirates',
    'uk': 'United Kingdom of Great Britain and Northern Ireland',   

}

# 3) Aplicar excepciones → columna intermedia
df_proyPibtras["pais_db"] = df_proyPibtras["pais_norm"].map(
    lambda x: exceptions[x] if x in exceptions else x
)

# 4) Mapear con la dimensión de países cargada previamente
# (asegúrate de tener dim_paises = {nombre_en.lower(): codigo})
df_proyPibtras["pais_id"] = (
    df_proyPibtras["pais_db"]
      .str.strip()
      .str.lower()
      .map(dim_paises)
)

# 5) Listar países no mapeados
no_map = df_proyPibtras.loc[df_proyPibtras["pais_id"].isna(), "Pais/ Region"].unique()
print("⚠️ Países sin mapeo:", no_map)

df_proyPibtras

⚠️ Países sin mapeo: ['OECD' 'North America' 'South America' 'Europe' 'Middle East & Africa'
 'Asia' 'Advanced Asia' 'ASEAN' 'Oceania']


,Pais/ Region,Escenario incremento T°,Escenario desastres,Impacto en PIB,pais_norm,pais_db,pais_id
0,World,Incremento bajo 2°C (Acuerdo París),Sin incremento en desastres,-0.5,world,world,WO
1,OECD,Incremento bajo 2°C (Acuerdo París),Sin incremento en desastres,-0.4,oecd,oecd,NaN
2,North America,Incremento bajo 2°C (Acuerdo París),Sin incremento en desastres,-0.5,north america,north america,NaN
3,South America,Incremento bajo 2°C (Acuerdo París),Sin incremento en desastres,-0.4,south america,south america,NaN
4,Europe,Incremento bajo 2°C (Acuerdo París),Sin incremento en desastres,-0.2,europe,europe,NaN
...,...,...,...,...,...,...,...
691,UAE,Incremento 3.2°C,Extremo incremento en desastres (x10),-33.7,uae,United Arab Emirates,AE
692,UK,Incremento 3.2°C,Extremo incremento en desastres (x10),-8.7,uk,United Kingdom of Great Britain and Northern I...,GB
693,Ukraine,Incremento 3.2°C,Extremo incremento en desastres (x10),-5.6,ukraine,ukraine,UA
694,US,Incremento 3.2°C,Extremo incremento en desastres (x10),-9.2,us,united states of america,US


In [3]:
df_proyPibtras = df_proyPibtras.loc[df_proyPibtras["pais_id"].notna()].copy()


# Diccionario de mapeo: (Escenario T°, Escenario desastres) → indicador_id
indicadores_map = {
    ("Incremento bajo 2°C (Acuerdo París)", "Sin incremento en desastres"): 26,
    ("Incremento bajo 2°C (Acuerdo París)", "Mediano incremento en desastres (x5)"): 27,
    ("Incremento bajo 2°C (Acuerdo París)", "Extremo incremento en desastres (x10)"): 28,

    ("Incremento 2.0°C", "Sin incremento en desastres"): 29,
    ("Incremento 2.0°C", "Mediano incremento en desastres (x5)"): 30,
    ("Incremento 2.0°C", "Extremo incremento en desastres (x10)"): 31,

    ("Incremento 2.6°C", "Sin incremento en desastres"): 32,
    ("Incremento 2.6°C", "Mediano incremento en desastres (x5)"): 33,
    ("Incremento 2.6°C", "Extremo incremento en desastres (x10)"): 34,

    ("Incremento 3.2°C", "Sin incremento en desastres"): 35,
    ("Incremento 3.2°C", "Mediano incremento en desastres (x5)"): 36,
    ("Incremento 3.2°C", "Extremo incremento en desastres (x10)"): 37,
}


df_proyPibtras["indicador_id"] = df_proyPibtras.apply(
    lambda row: indicadores_map.get(
        (row["Escenario incremento T°"], row["Escenario desastres"])
    ),
    axis=1
)


# Crear columna año si corresponde (ejemplo: todos 2024, o derivado de otra variable)
df_proyPibtras["anio"] = 2020 

# Seleccionar columnas finales
df_hechos = df_proyPibtras[["anio", "Impacto en PIB", "pais_id", "indicador_id"]].rename(
    columns={"Impacto en PIB": "valor"}
)

print(df_hechos.head(100))


     anio  valor pais_id  indicador_id
0    2020 -0.500      WO            26
10   2020 -0.400      AR            26
11   2020 -0.500      AU            26
12   2020  0.001      AT            26
13   2020 -0.100      BE            26
..    ...    ...     ...           ...
113  2020  0.000      UA            29
114  2020 -0.900      US            29
115  2020 -0.900      VE            29
116  2020 -1.700      WO            32
126  2020 -0.900      AR            32

[100 rows x 4 columns]


In [4]:
sql_insert = """
INSERT INTO Hechos (anio, valor, pais_id, indicador_id)
VALUES (%s, %s, %s, %s)
"""

cursor.executemany(sql_insert, df_hechos.values.tolist())
conexion.commit()
print(f"✅ Se insertaron {cursor.rowcount} registros en la tabla Hechos")

✅ Se insertaron 588 registros en la tabla Hechos
